## noop_notebook
Do-little harness notebook for `error_testing`. Does NOTHING except optionally raise, selected by the run-time `fail_labels` parameter.

- `use_steplog=true` → opens a `StepLog` row; a raise is a **logged** failure (`step.fail(e); raise`).
- `use_steplog=false` → no step row; a raise is a **pre-logging** failure; the clean path exits via `dbutils.notebook.exit()` **outside** any try.

Spec: `_dev_planning/design_docs/error_testing_harness_design.md`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# Params. notebook_init already injected AUDIT, PIPELINE_RUN_ID, StepLog, Utils, F, datetime, STATUS_*.
dbutils.widgets.text("label", "")
dbutils.widgets.text("step_sequence", "0")
dbutils.widgets.text("use_steplog", "false")   # "true" | "false"
dbutils.widgets.text("fail_labels", "")         # CSV of labels to fail this run
LABEL         = dbutils.widgets.get("label").strip()
STEP_SEQUENCE = int(dbutils.widgets.get("step_sequence"))
USE_STEPLOG   = dbutils.widgets.get("use_steplog").strip().lower() == "true"
# Split CSV -> trimmed non-empty labels; this task fails iff its LABEL is in the set.
FAIL_LABELS   = [s.strip() for s in dbutils.widgets.get("fail_labels").split(",") if s.strip()]
SHOULD_FAIL   = LABEL in FAIL_LABELS
print(f"noop_notebook: label={LABEL} use_steplog={USE_STEPLOG} should_fail={SHOULD_FAIL} fail_labels={FAIL_LABELS}")

In [ ]:
# Pre-logging branch (use_steplog=false): NO StepLog opened, so a raise writes NO
# pipeline_step_log row — the 2026-06-01 masking class. The clean no-op path exits
# OUTSIDE any try (dbutils.notebook.exit raises an ordinary exception that except would swallow).
if not USE_STEPLOG:
    print(f"noop_notebook[{LABEL}]: pre-logging branch (no StepLog)")
    if SHOULD_FAIL:
        raise RuntimeError(f"noop_notebook[{LABEL}]: injected PRE-LOGGING failure (fail_labels={FAIL_LABELS})")
    dbutils.notebook.exit(f"noop_notebook[{LABEL}]: clean no-op exit")

In [ ]:
# Logged branch (use_steplog=true): open the pipeline_step_log RUNNING row.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "error_testing",
    target_table    = None,
)
print(f"noop_notebook[{LABEL}]: step_log_id={step.step_log_id}")

In [ ]:
# Do-little work cell; a raise here is a LOGGED failure (step row closes 'failed').
try:
    step.rows_read = 0
    if SHOULD_FAIL:
        raise RuntimeError(f"noop_notebook[{LABEL}]: injected LOGGED failure (fail_labels={FAIL_LABELS})")
    step.rows_written = 0
    step.succeed()
    print(f"noop_notebook[{LABEL}]: closed SUCCEEDED")
except Exception as e:
    step.fail(e); raise